In [1]:
%%bash
ref_dir=data/toy_ref_read/toy_human_ref
pyroe make-splici ${ref_dir}/fasta/genome.fa ${ref_dir}/genes/genes.gtf 90 splici_rl90_ref

+ which python


/Users/ashleyosuna/miniconda3/envs/pyroe-env/bin/python


+ python3 --version


Python 3.10.19


+ python3 -c 'import numpy, pandas, matplotlib, scanpy; print(numpy.__version__, pandas.__version__, matplotlib.__version__, scanpy.__version__)'


1.23.5 1.5.3 3.7.3 1.9.8


+ which pyroe


/Users/ashleyosuna/miniconda3/envs/pyroe-env/bin/pyroe


+ ref_dir=data/toy_ref_read/toy_human_ref
+ pyroe make-splici data/toy_ref_read/toy_human_ref/fasta/genome.fa data/toy_ref_read/toy_human_ref/genes/genes.gtf 90 splici_rl90_ref


/Users/ashleyosuna/miniconda3/envs/pyroe-env/lib/python3.10/site-packages/pyranges/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [1]:
%%bash
salmon index -t $(ls splici_rl90_ref/*\.fa) -i salmon_index -p 8

Version Server Response: Not Found
index ["salmon_index"] did not previously exist  . . . creating it
[2025-11-17 07:36:05.010] [jLog] [warning] The salmon index is being built without any decoy sequences.  It is recommended that decoy sequence (either computed auxiliary decoy sequence or the genome of the organism) be provided during indexing. Further details can be found at https://salmon.readthedocs.io/en/latest/salmon.html#preparing-transcriptome-indices-mapping-based-mode.
[2025-11-17 07:36:05.011] [jLog] [info] building index
out : salmon_index
[2025-11-17 07:36:05.011] [puff::index::jointLog] [info] Running fixFasta

[Step 1 of 4] : counting k-mers

[2025-11-17 07:36:05.085] [puff::index::jointLog] [info] Replaced 0 non-ATCG nucleotides
[2025-11-17 07:36:05.085] [puff::index::jointLog] [info] Clipped poly-A tails from 8 transcripts
wrote 337 cleaned references
[2025-11-17 07:36:05.091] [puff::index::jointLog] [info] Filter size not provided; estimating from number of distinct k-

Threads = 8
Vertex length = 31
Hash functions = 5
Filter size = 67108864
Capacity = 2
Files: 
salmon_index/ref_k31_fixed.fa
--------------------------------------------------------------------------------
Round 0, 0:67108864
Pass	Filling	Filtering
1	0	0	
2	0	0
True junctions count = 9520
False junctions count = 10731
Hash table size = 20251
Candidate marks count = 68082
--------------------------------------------------------------------------------
Reallocating bifurcations time: 0
True marks count: 52427
Edges construction time: 0
--------------------------------------------------------------------------------
Distinct junctions = 9520



TwoPaCo::buildGraphMain:: allocated with scalable_malloc; freeing.
TwoPaCo::buildGraphMain:: Calling scalable_allocation_command(TBBMALLOC_CLEAN_ALL_BUFFERS, 0);
allowedIn: 18
Max Junction ID: 9602
seen.size():76825 kmerInfo.size():9603
approximateContigTotalLength: 770541
counters for complex kmers:
(prec>1 & succ>1)=1549 | (succ>1 & isStart)=1 | (prec>1 & isEnd)=1 | (isStart & isEnd)=0
contig count: 15290 element count: 2453489 complex nodes: 1551
# of ones in rank vector: 15289
[2025-11-17 07:36:05.745] [puff::index::jointLog] [info] Starting the Pufferfish indexing by reading the GFA binary file.
[2025-11-17 07:36:05.745] [puff::index::jointLog] [info] Setting the index/BinaryGfa directory salmon_index
size = 2453489
-----------------------------------------
| Loading contigs | Time = 514.04 us
-----------------------------------------
size = 2453489
-----------------------------------------
| Loading contig boundaries | Time = 186.17 us
-----------------------------------------
Nu

for info, total work write each  : 2.331    total work inram from level 3 : 4.322  total work raw : 25.000 
Bitarray        10458112  bits (100.00 %)   (array + ranks )
final hash             0  bits (0.00 %) (nb in final hash 0)


In [14]:
%%bash
# Mapping and quantification
fastq_dir="data/toy_ref_read/toy_read_fastq"
reads1_pat="selected_R1_reads.fastq"
reads2_pat="selected_R2_reads.fastq"

# Mapping
salmon alevin \
-i salmon_index \
-l ISR \
-1 $fastq_dir/$reads1_pat \
-2 $fastq_dir/$reads2_pat \
-p 8 \
-o salmon_alevin \
--chromiumV3 \
--sketch

usage: paste [-s] [-d delimiters] file ...
usage: paste [-s] [-d delimiters] file ...


Version Server Response: Not Found
[2025-11-17 07:54:12.181] [alevinLog] [info] currently, --sketch implies --rad. Running in alignment-only mode (will write a RAD output).
Logs will be written to salmon_alevin/logs
[2025-11-17 07:54:12.181] [alevinLog] [info] The --rad flag was passed to alevin. The reads will be selectively aligned and the output written to a RAD file.Arguments passed that correspond to other processing steps will be ignored
[2025-11-17 07:54:12.181] [alevinLog] [info] The --sketch flag was passed; the alignment will be run in sketch mode.
[2025-11-17 07:54:12.188] [jointLog] [info] setting maxHashResizeThreads to 8
[2025-11-17 07:54:12.188] [jointLog] [info] Fragment incompatibility prior below threshold.  Incompatible fragments will be ignored.
[2025-11-17 07:54:12.188] [jointLog] [info] The --mimicBT2, --mimicStrictBT2 and --hardFilter flags imply mapping validation (--validateMappings). Enabling mapping validation.
[2025-11-17 07:54:12.188] [jointLog] [info] Usag

In [15]:
%%bash
# Cell barcode correction
alevin-fry generate-permit-list \
-u data/3M-february-2018.txt \
-d fw \
-i salmon_alevin \
-o alevin_fry_gpl

# Filter mapping information
alevin-fry collate \
-i alevin_fry_gpl \
-r salmon_alevin \
-t 8

# UMI resolution + quantification
alevin-fry quant -r cr-like \
-m $(ls splici_rl90_ref/*3col.tsv) \
-i alevin_fry_gpl \
-o alevin_fry_quant \
-t 8

bash: line 2: alevin-fry: command not found
bash: line 9: alevin-fry: command not found
bash: line 15: alevin-fry: command not found


CalledProcessError: Command 'b'# Cell barcode correction\nalevin-fry generate-permit-list \\\n-u 3M-february-2018.txt \\\n-d fw \\\n-i salmon_alevin \\\n-o alevin_fry_gpl\n\n# Filter mapping information\nalevin-fry collate \\\n-i alevin_fry_gpl \\\n-r salmon_alevin \\\n-t 8\n\n# UMI resolution + quantification\nalevin-fry quant -r cr-like \\\n-m $(ls splici_rl90_ref/*3col.tsv) \\\n-i alevin_fry_gpl \\\n-o alevin_fry_quant \\\n-t 8\n'' returned non-zero exit status 127.